In [7]:
import pandas as pd
import numpy as np
import math
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, log_loss
df = pd.read_csv('Car_Prices.csv')

In [8]:
# Handle Outliers
df = df[(df['Price'] >= 500) & (df['Price'] <= 150000)]
df = df[(df['kms_driven'] >= 0) & (df['kms_driven'] <= 1000000)]

# Handle High Cardinality in 'Model'
top_models = df['Model'].value_counts().nlargest(100).index
df['Model'] = df['Model'].where(df['Model'].isin(top_models), 'Other')

In [9]:
# Convert Categorical to Numeric
cols_to_convert = ['Manufacturer', 'Model', 'Category', 'Leather interior', 'Drive wheels', 'Wheel', 'Fuel type', 'Gear box type']
for col in cols_to_convert:
    df[col] = df[col].astype('category').cat.codes

# Min-Max Normalization
features_to_scale = ['Tax', 'Car age', 'Airbags', 'Total_doors', 'Cylinders', 'Engine volume', 'kms_driven']
for col in features_to_scale:
    df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

In [10]:
#TARGET PREPARATION
df_encoded = df.copy()

# Rearrange numerical Price into 5 categorical intervals (Requirement 2)
df_encoded['Price_Category'] = pd.cut(df_encoded['Price'], bins=5, labels=False)
y = df_encoded['Price_Category']

#FEATURE SELECTION
# Core Feature Set: Using primary continuous and specific categorical attributes
core_features = ['Tax', 'Airbags', 'kms_driven', 'Car age', 'Engine volume']
X_core = df_encoded[core_features]

# Comprehensive Feature Set: Using all processed attributes including Manufacturer
X_comprehensive = df_encoded.drop(['Price', 'Price_Category', 'Model'], axis=1)

print(f"Initial setup complete. Target binned into {y.nunique()} intervals.")
print("\nPrice Category Distribution:")
dist = y.value_counts().sort_index()
for cat, count in dist.items():
    pct = count / len(y) * 100
    print(f"  Category {int(cat)}: {count} samples ({pct:.1f}%)")

Initial setup complete. Target binned into 5 intervals.

Price Category Distribution:
  Category 0: 12160 samples (82.7%)
  Category 1: 2079 samples (14.1%)
  Category 2: 326 samples (2.2%)
  Category 3: 99 samples (0.7%)
  Category 4: 37 samples (0.3%)


In [11]:
# INITIAL ANN MODEL WITH 10-FOLD CROSS VALIDATION 
kf = KFold(n_splits=10, shuffle=True, random_state=42)
initial_accuracies = []
initial_losses = []

print("Executing Initial 10-Fold Cross-Validation...\n")

for fold, (train_idx, test_idx) in enumerate(kf.split(X_core), 1):
    X_train, X_test = X_core.iloc[train_idx], X_core.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Initial ANN Architecture: Single hidden layer of 10 neurons
    mlp_initial = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', max_iter=1000, random_state=42)
    mlp_initial.fit(X_train, y_train)
    
    # Evaluation
    preds = mlp_initial.predict(X_test)
    acc = accuracy_score(y_test, preds)
    loss = log_loss(y_test, mlp_initial.predict_proba(X_test))
    
    initial_accuracies.append(acc)
    initial_losses.append(loss)
    print(f"Fold {fold}: Accuracy = {acc:.4f}, Cost Function (Loss) = {loss:.4f}")

print(f"\nAverage Initial Accuracy: {np.mean(initial_accuracies):.4f}")
print(f"Average Initial Loss: {np.mean(initial_losses):.4f}")

Executing Initial 10-Fold Cross-Validation...

Fold 1: Accuracy = 0.8491, Cost Function (Loss) = 0.4190
Fold 2: Accuracy = 0.8497, Cost Function (Loss) = 0.3833
Fold 3: Accuracy = 0.8238, Cost Function (Loss) = 0.4652
Fold 4: Accuracy = 0.8347, Cost Function (Loss) = 0.4255
Fold 5: Accuracy = 0.8449, Cost Function (Loss) = 0.3806
Fold 6: Accuracy = 0.8483, Cost Function (Loss) = 0.3992
Fold 7: Accuracy = 0.8422, Cost Function (Loss) = 0.4188
Fold 8: Accuracy = 0.8633, Cost Function (Loss) = 0.3574
Fold 9: Accuracy = 0.8490, Cost Function (Loss) = 0.3766
Fold 10: Accuracy = 0.8476, Cost Function (Loss) = 0.3970

Average Initial Accuracy: 0.8452
Average Initial Loss: 0.4023


In [12]:
#OPTIMIZED ANN MODEL PERFORMANCE IMPROVEMENT
optimized_accuracies = []
optimized_losses = []

print("Executing Optimized 10-Fold Cross-Validation...\n")

for fold, (train_idx, test_idx) in enumerate(kf.split(X_comprehensive), 1):
    X_train, X_test = X_comprehensive.iloc[train_idx], X_comprehensive.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Measures: Rearranged dataset (more features), increased ANN structure (160, 80), 
    # and adjusted learning rate (0.0005) for precision.
    mlp_optimized = MLPClassifier(
        hidden_layer_sizes=(160, 80), 
        activation='relu', 
        learning_rate_init=0.0005, 
        max_iter=1000, 
        random_state=42
    )
    
    mlp_optimized.fit(X_train, y_train)
    
    # Evaluation
    preds = mlp_optimized.predict(X_test)
    acc = accuracy_score(y_test, preds)
    loss = log_loss(y_test, mlp_optimized.predict_proba(X_test))    
    
    optimized_accuracies.append(acc)
    optimized_losses.append(loss)
    print(f"Fold {fold}: Accuracy = {acc:.4f}, Cost Function (Loss) = {loss:.4f}")

print(f"\nFinal Optimized Average Accuracy: {np.mean(optimized_accuracies):.4f}")
print(f"Final Optimized Average Loss: {np.mean(optimized_losses):.4f}")

Executing Optimized 10-Fold Cross-Validation...

Fold 1: Accuracy = 0.9082, Cost Function (Loss) = 0.2759
Fold 2: Accuracy = 0.8966, Cost Function (Loss) = 0.2883
Fold 3: Accuracy = 0.8864, Cost Function (Loss) = 0.3368
Fold 4: Accuracy = 0.8844, Cost Function (Loss) = 0.3043
Fold 5: Accuracy = 0.9020, Cost Function (Loss) = 0.2496
Fold 6: Accuracy = 0.9088, Cost Function (Loss) = 0.2825
Fold 7: Accuracy = 0.8830, Cost Function (Loss) = 0.3043
Fold 8: Accuracy = 0.9177, Cost Function (Loss) = 0.2281
Fold 9: Accuracy = 0.9109, Cost Function (Loss) = 0.2450
Fold 10: Accuracy = 0.8891, Cost Function (Loss) = 0.2959

Final Optimized Average Accuracy: 0.8987
Final Optimized Average Loss: 0.2811
